In [1]:
!conda install ray[cgraph]

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-64
- 

In [2]:
import ray
import torch
import time
import os
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to the existing Ray cluster
ray.init(address="auto", ignore_reinit_error=True)

# 1 GB float32 tensor
TENSOR_SIZE_BYTES = 1 * 1024**3 
TENSOR_SHAPE = (250_000_000,)

print(f"Ray Cluster Resources: {ray.available_resources()}")
print(f"Testing with Payload Size: {TENSOR_SIZE_BYTES / 1e9:.2f} GB")

# Set up plotting style for academic/HPC reporting
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

2026-06-08 14:02:44,526	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-06-08 14:02:44,560	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://10.0.1.14:8265 


Ray Cluster Resources: {'object_store_memory': 84051620657.0, 'memory': 196120448207.0, 'node:10.0.1.10': 1.0, 'CPU': 88.0, 'node:__internal_head__': 1.0, 'accelerator_type:RTX': 2.0, 'GPU': 2.0, 'node:10.0.1.14': 1.0, 'accelerator_type:A6000': 2.0, 'node:10.0.1.34': 1.0, 'node:10.0.1.18': 1.0, 'node:10.0.1.38': 1.0}
Testing with Payload Size: 1.07 GB


/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [4]:
@ray.remote(num_gpus=1)
class Sender:
    def __init__(self, shape):
        self.shape = shape
        self.tensor = torch.ones(self.shape, dtype=torch.float32, device="cuda")
        
    def get_data(self, trigger_input): # <-- FIX: Added dummy parameter
        return self.tensor

@ray.remote(num_gpus=1)
class Receiver:
    def __init__(self):
        self.sink = None

    def consume_data(self, tensor):
        self.sink = tensor
        return True

# Instantiate the actors
sender = Sender.remote(TENSOR_SHAPE)
receiver = Receiver.remote()

# Warm up GPUs (pass a 0 as the dummy trigger)
ray.get(receiver.consume_data.remote(sender.get_data.remote(0)))
print("Actors initialized and warmed up.")

Actors initialized and warmed up.


In [5]:
# # --- Baseline: Ray Object Store ---
# def benchmark_object_store(iterations=20):
#     records = []
#     print(f"Running Baseline (Object Store) for {iterations} iterations...")
    
#     for i in range(iterations):
#         start_time = time.perf_counter()
        
#         # Standard Ray object store transfer (pass dummy '0')
#         data_ref = sender.get_data.remote(0) 
#         ready_ref = receiver.consume_data.remote(data_ref)
#         ray.get(ready_ref) 
        
#         end_time = time.perf_counter()
#         iteration_time = end_time - start_time
        
#         records.append({
#             "Method": "Ray Object Store (TCP/CPU)",
#             "Iteration": i + 1,
#             "Latency (ms)": iteration_time * 1000,
#             "Throughput (GB/s)": (TENSOR_SIZE_BYTES / 1e9) / iteration_time
#         })
        
#     return pd.DataFrame(records)

# df_baseline = benchmark_object_store(iterations=20)

In [7]:
# Force both actors onto the same node for testing
sender = Sender.options(scheduling_strategy=ray.util.scheduling_strategies.NodeAffinitySchedulingStrategy(
    node_id=ray.get_runtime_context().get_node_id(), soft=False)).remote(TENSOR_SHAPE)

receiver = Receiver.options(scheduling_strategy=ray.util.scheduling_strategies.NodeAffinitySchedulingStrategy(
    node_id=ray.get_runtime_context().get_node_id(), soft=False)).remote()

In [ ]:
from ray.dag.input_node import InputNode

# Define and compile the graph
with InputNode() as inp:
    # FIX: Pass 'inp' to anchor the start of the computation graph
    data = sender.get_data.bind(inp) 
    result = receiver.consume_data.bind(data)

compiled_dag = result.experimental_compile()
print(compiled_dag.visualize(format="ascii", view=True))

# Warmup (pass a dummy trigger value)
# compiled_dag.execute(0) 

def benchmark_compiled_graph(iterations=1):
    records = []
    print(f"Running Compiled Graph (NCCL) for {iterations} iterations...")
    
    for i in tqdm(range(iterations)):
        start_time = time.perf_counter()
        
        # FIX: Pass 'i' as the dummy input to trigger execution
        ray.get(compiled_dag.execute(i)) 
        
        end_time = time.perf_counter()
        iteration_time = end_time - start_time
        
        records.append({
            "Method": "Compiled Graph + RDT (NCCL/RDMA)",
            "Iteration": i + 1,
            "Latency (ms)": iteration_time * 1000,
            "Throughput (GB/s)": (TENSOR_SIZE_BYTES / 1e9) / iteration_time
        })
        
    return pd.DataFrame(records)

df_nccl = benchmark_compiled_graph(iterations=1)

# Merge datasets for plotting
df_results = pd.concat([df_baseline, df_nccl], ignore_index=True)
print("Benchmarking complete. Data accumulated.")

In [ ]:
def plot_benchmark_results(df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'GPU-to-GPU Transfer Performance (Payload: {TENSOR_SIZE_BYTES / 1e9:.1f} GB Tensor)', fontweight='bold')

    # Plot 1: Throughput Distribution (Violin Plot)
    # This highlights stability and variance across iterations
    sns.violinplot(
        data=df, 
        x="Method", 
        y="Throughput (GB/s)", 
        ax=axes[0],
        palette=["#e74c3c", "#2ecc71"],
        inner="quartile"
    )
    axes[0].set_title("Throughput Distribution per Iteration")
    axes[0].set_ylabel("Throughput (GB/s)")
    axes[0].set_xlabel("")
    
    # Add a horizontal dashed line representing the theoretical PCIe 4.0 limit (~31.5 GB/s)
    axes[0].axhline(y=31.5, color='gray', linestyle='--', label='PCIe 4.0 x16 Limit (~31.5 GB/s)')
    axes[0].legend()

    # Plot 2: Average Latency (Bar Plot)
    # Shows the raw time cost of a single transfer
    sns.barplot(
        data=df, 
        x="Method", 
        y="Latency (ms)", 
        ax=axes[1],
        palette=["#e74c3c", "#2ecc71"],
        capsize=.1,
        err_kws={'linewidth': 2}
    )
    axes[1].set_title("Average Transfer Latency (Lower is Better)")
    axes[1].set_ylabel("Latency (ms)")
    axes[1].set_xlabel("")

    plt.tight_layout()
    plt.show()

    # Output the raw summary statistics for the report
    print("\n--- Summary Statistics ---")
    summary = df.groupby("Method")[["Throughput (GB/s)", "Latency (ms)"]].agg(['mean', 'std', 'min', 'max'])
    print(summary.to_string())

plot_benchmark_results(df_results)